<h1>Installing Libraries</h1>

In [27]:
from IPython.display import clear_output
!pip install beautifulsoup4
!pip install lxml
!pip install requests
clear_output()

<h1>Setting Up Selenium</h1>

In [28]:
!wget -q https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb
!dpkg -i google-chrome-stable_current_amd64.deb > /dev/null 2>&1
!apt-get -f install -y > /dev/null 2>&1  # Fix any dependency errors automatically

# --- STEP 2: INSTALL SELENIUM ---
!pip install -q selenium

# --- STEP 3: RUN SELENIUM ---
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from bs4 import BeautifulSoup
import time

def run_scraper_fixed(url):
    print("⚙️ Setting up Chrome...")

    options = webdriver.ChromeOptions()
    options.add_argument('--headless')
    options.add_argument('--no-sandbox')
    options.add_argument('--disable-dev-shm-usage')

    # CRITICAL: Use the official binary we just installed
    options.binary_location = "/usr/bin/google-chrome"

    # Selenium 4.x will automatically download the matching driver for us!
    driver = webdriver.Chrome(options=options)

    try:
        print(f"Navigating to {url}...")
        driver.get(url)

        # Give it a moment to render JS
        time.sleep(3)

        # Get the soup
        soup = BeautifulSoup(driver.page_source, 'lxml')
        return soup, driver.title

    except Exception as e:
        print(f"❌ Error: {e}")
        return None, None
    finally:
        driver.quit()
clear_output()

<h1>Importing Libraries</h1>

In [29]:
from bs4 import BeautifulSoup
import requests
import pandas as pd

<h1>Finding Openings</h1>

Parsing Function

In [30]:
def parse_adndiginet_jobs(soup):
  jobs = []
  tags = soup.find_all('h2', class_="text-lg font-semibold text-gray-900")
  for tag in tags:
    jobs.append(tag.text.strip())
  return jobs

def parse_robi_jobs(soup):
  jobs = []
  container = soup.find('div', id='cdk-describedby-message-container')
  if container:
    for job_div in container.find_all('div', recursive=False):
      raw_text = job_div.get_text(strip=True)
      clean_text = raw_text.split(' (Ref')[0]
      clean_text = clean_text.split(', Ref')[0]
      clean_text = clean_text.split(', 100%')[0]
      clean_text = clean_text.strip(', ')
      jobs.append(clean_text)
  return jobs

def parse_banglalink_jobs(soup):
  jobs = []
  tags = soup.find_all('span',class_="job-tile__title")
  for tag in tags:
    jobs.append(tag.text.strip())
  return jobs

Company Sites and URL:

In [31]:
target_sites = [
    {
        "company": "ADN DigiNet",
        "url": "https://adndiginet.com/career",
        "parser": parse_adndiginet_jobs
    },
    {
        "company": "Robi",
        "url": "https://robicareer.com/job-portal/jobs/all-jobs",
        "parser": parse_robi_jobs
    },
    {
        "company": "Banglalink",
        "url": "https://fa-esth-saasfaprod1.fa.ocs.oraclecloud.com/hcmUI/CandidateExperience/en/sites/CX_1/jobs?location=Bangladesh&locationId=300000000440425&locationLevel=country&mode=job-location",
        "parser": parse_banglalink_jobs
    }
]

Main Loop:

In [32]:
all_job_data = []

print("Starting My Job Scraper...\n")

for site in target_sites:
    print(f"Checking {site['company']}...")
    soup, page_title = run_scraper_fixed(site['url'])

    if soup:
        found_jobs = site['parser'](soup)
        print(f"   -> Found {len(found_jobs)} jobs.")
        for job in found_jobs:
            all_job_data.append({
                "Company Name": site['company'],
                "Job Title": job,
                "Url": site['url']
            })
    else:
        print(f"   -> Failed to scrape {site['company']}")

Starting My Job Scraper...

Checking ADN DigiNet...
⚙️ Setting up Chrome...
Navigating to https://adndiginet.com/career...
   -> Found 4 jobs.
Checking Robi...
⚙️ Setting up Chrome...
Navigating to https://robicareer.com/job-portal/jobs/all-jobs...
   -> Found 2 jobs.
Checking Banglalink...
⚙️ Setting up Chrome...
Navigating to https://fa-esth-saasfaprod1.fa.ocs.oraclecloud.com/hcmUI/CandidateExperience/en/sites/CX_1/jobs?location=Bangladesh&locationId=300000000440425&locationLevel=country&mode=job-location...
   -> Found 4 jobs.


<h1>Making Table</h1>

In [33]:
df = pd.DataFrame(all_job_data)
print("\nFinal Data:")
print(df)


Final Data:
  Company Name                                          Job Title  \
0  ADN DigiNet                                    DevOps Engineer   
1  ADN DigiNet                    DevOps Engineer (Cybersecurity)   
2  ADN DigiNet                                   TechOps Engineer   
3  ADN DigiNet                          Site Reliability Engineer   
4         Robi  Manager or Senior Manager, Digitalization & Au...   
5         Robi   Junior Software Engineer, RedDot Digital Limited   
6   Banglalink          Digital Solutions & Service Lead Engineer   
7   Banglalink                               ICT Business Manager   
8   Banglalink                        ICT Business Senior Manager   
9   Banglalink      Senior Corporate Account Manager, Key Segment   

                                                 Url  
0                      https://adndiginet.com/career  
1                      https://adndiginet.com/career  
2                      https://adndiginet.com/career  
3      